<a href="https://colab.research.google.com/github/luckestocks/pdf-chatbot-capstone/blob/main/pdf_chatbot_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# WORKING GROQ MODELS (as of March 2026)
# llama-3.1-8b-instant   — fast, free, use this as default
# llama-3.3-70b-versatile — bigger, smarter, slower (use for complex tasks)
# mixtral-8x7b-32768      — alternative if llama has issues

In [ ]:
# 🤖 AI-Powered PDF Chatbot — Research Grade
## Capstone Project | RAG Pipeline

**Tech Stack:** LangChain · ChromaDB · Groq (Llama-3.1) · RAGAS · Streamlit

**Pipelines:**
- Pipeline 1: Document Ingestion
- Pipeline 2: Query & Response
- Pipeline 3: Evaluation & Experiments

In [3]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
import subprocess
import sys

packages = [
    "pypdf2",
    "langchain",
    "langchain-community",
    "sentence-transformers",
    "chromadb",
    "groq",
    "pdfplumber",
    "rank-bm25",
    "ragas"
]

print("Installing packages...\n")
for package in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package, "-q", "--no-warn-conflicts"],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {package}")
    else:
        print(f"  ❌ {package} — {result.stderr.strip()}")

print("\nAll packages processed!")

Installing packages...

  ✅ pypdf2
  ✅ langchain
  ✅ langchain-community
  ✅ sentence-transformers
  ✅ chromadb
  ✅ groq
  ✅ pdfplumber
  ✅ rank-bm25
  ✅ ragas

All packages processed!


In [4]:
# ============================================================
# CELL 2 — Verify All Imports
# ============================================================
try:
    import PyPDF2
    import langchain
    import chromadb
    import pdfplumber
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi
    from groq import Groq
    print("✅ All libraries imported successfully!")
    print("✅ Environment is ready to build!")
except ImportError as e:
    print(f"❌ Missing library: {e}")
    print("Re-run Cell 1 to fix")

✅ All libraries imported successfully!
✅ Environment is ready to build!


In [7]:
# ============================================================
# CELL 3 — Connect to Groq (Llama-3.1)
# ============================================================
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

# Quick test
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say: Groq connection successful!"}]
)

print("✅ " + response.choices[0].message.content)



✅ It seems like we have established a connection with Groq, a company that specializes in AI hardware and software solutions. What would you like to do next?


In [8]:
# ============================================================
# CELL 4 — Upload PDF to Colab
# ============================================================
from google.colab import files

print("A file picker will appear below...")
print("Select your rag_paper.pdf file from your laptop\n")

uploaded = files.upload()

# Show what was uploaded
for filename in uploaded.keys():
    print(f"✅ Successfully uploaded: {filename}")
    print(f"   Size: {len(uploaded[filename]):,} bytes")

A file picker will appear below...
Select your rag_paper.pdf file from your laptop



Saving rag_paper.pdf to rag_paper.pdf
✅ Successfully uploaded: rag_paper.pdf
   Size: 885,323 bytes


In [9]:
# ============================================================
# CELL 5 — Extract Text from PDF
# ============================================================
import PyPDF2

# Open and read the PDF
pdf_file = open("rag_paper.pdf", "rb")  # rb = read binary
reader = PyPDF2.PdfReader(pdf_file)

# How many pages?
total_pages = len(reader.pages)
print(f"✅ PDF loaded successfully!")
print(f"📄 Total pages: {total_pages}")
print()

# Extract text from all pages
full_text = ""
for page_num in range(total_pages):
    page = reader.pages[page_num]
    text = page.extract_text()
    full_text += text

print(f"📝 Total characters extracted: {len(full_text):,}")
print()

# Preview first 500 characters so we can see it worked
print("--- PREVIEW OF EXTRACTED TEXT ---")
print(full_text[:500])
print("...")

✅ PDF loaded successfully!
📄 Total pages: 19

📝 Total characters extracted: 69,113

--- PREVIEW OF EXTRACTED TEXT ---
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewisyz, Ethan Perez?,
Aleksandra Piktusy, Fabio Petroniy, Vladimir Karpukhiny, Naman Goyaly, Heinrich Küttlery,
Mike Lewisy, Wen-tau Yihy, Tim Rocktäschelyz, Sebastian Riedelyz, Douwe Kielay
yFacebook AI Research;zUniversity College London;?New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁ
...


In [11]:
!pip install langchain-text-splitters -q

In [12]:
# ============================================================
# CELL 6 — Split Text into Chunks
# ============================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create our splitter with settings from our config
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,        # each chunk = max 800 characters
    chunk_overlap=100,     # chunks share 100 characters with neighbours
                           # overlap helps so answers don't get cut off at edges
    separators=["\n\n", "\n", ".", " ", ""]  # try to split at natural breaks
)

# Split the full text into chunks
chunks = text_splitter.create_documents([full_text])

# Results
print(f"✅ Chunking complete!")
print(f"📄 Original text: {len(full_text):,} characters")
print(f"✂️  Total chunks created: {len(chunks)}")
print(f"📏 Average chunk size: {len(full_text) // len(chunks)} characters")
print()

# Preview first 3 chunks so we can see what they look like
print("--- PREVIEW: FIRST 3 CHUNKS ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(chunk.page_content)
    print("-" * 50)

✅ Chunking complete!
📄 Original text: 69,113 characters
✂️  Total chunks created: 102
📏 Average chunk size: 677 characters

--- PREVIEW: FIRST 3 CHUNKS ---

🔹 Chunk 1 (766 chars):
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewisyz, Ethan Perez?,
Aleksandra Piktusy, Fabio Petroniy, Vladimir Karpukhiny, Naman Goyaly, Heinrich Küttlery,
Mike Lewisy, Wen-tau Yihy, Tim Rocktäschelyz, Sebastian Riedelyz, Douwe Kielay
yFacebook AI Research;zUniversity College London;?New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
--------------------------------------------------

🔹

In [13]:
# ============================================================
# CELL 7 — Generate Embeddings
# ============================================================
from sentence_transformers import SentenceTransformer

print("⏳ Loading embedding model (bge-small-en)...")
print("   This downloads once and is cached after that\n")

# Load our free embedding model
embedding_model = SentenceTransformer("BAAI/bge-small-en")

print("✅ Embedding model loaded!\n")

# Extract just the text from our chunks
chunk_texts = [chunk.page_content for chunk in chunks]

print(f"⏳ Generating embeddings for {len(chunk_texts)} chunks...")
print("   Converting each chunk from text → numbers\n")

# Generate embeddings for all chunks
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,  # shows a progress bar
    batch_size=32
)

print(f"\n✅ Embeddings generated!")
print(f"📊 Shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} chunks each converted to {embeddings.shape[1]} numbers")
print(f"\n🔍 Preview of first chunk's embedding (first 5 numbers):")
print(f"   {embeddings[0][:5]}")
print(f"\n   These numbers represent the MEANING of Chunk 1 mathematically!")

⏳ Loading embedding model (bge-small-en)...
   This downloads once and is cached after that



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!

⏳ Generating embeddings for 102 chunks...
   Converting each chunk from text → numbers



Batches:   0%|          | 0/4 [00:00<?, ?it/s]


✅ Embeddings generated!
📊 Shape: (102, 384)
   → 102 chunks each converted to 384 numbers

🔍 Preview of first chunk's embedding (first 5 numbers):
   [-0.05706558  0.00146344 -0.010355    0.02674172  0.02093359]

   These numbers represent the MEANING of Chunk 1 mathematically!


In [14]:
# ============================================================
# CELL 8 — Store Embeddings in ChromaDB
# ============================================================
import chromadb
import uuid

print("⏳ Setting up ChromaDB vector store...\n")

# Create a ChromaDB client that stores data in memory
# (We will add persistent storage in a later phase)
chroma_client = chromadb.Client()

# Create a collection — think of this like a table in a database
# Delete if exists first (so we can re-run this cell cleanly)
try:
    chroma_client.delete_collection("rag_paper")
except:
    pass

collection = chroma_client.create_collection(
    name="rag_paper",
    metadata={"description": "RAG research paper chunks"}
)

print("✅ ChromaDB collection created!\n")

# Add all chunks and their embeddings to ChromaDB
print(f"⏳ Storing {len(chunks)} chunks in ChromaDB...")

collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],        # unique ID for each chunk
    embeddings=embeddings.tolist(),                  # the 384 numbers per chunk
    documents=chunk_texts,                           # the original text
    metadatas=[{"chunk_index": i, "source": "rag_paper.pdf"}
               for i in range(len(chunks))]          # extra info per chunk
)

print(f"✅ All chunks stored successfully!")
print(f"📦 Total items in ChromaDB: {collection.count()}")
print()

# Test it works — search for something!
print("🔍 Running a test search: 'what is retrieval augmented generation?'")
print()

test_query_embedding = embedding_model.encode(
    ["what is retrieval augmented generation?"]
).tolist()

results = collection.query(
    query_embeddings=test_query_embedding,
    n_results=3
)

print("--- TOP 3 MOST RELEVANT CHUNKS FOUND ---")
for i, doc in enumerate(results["documents"][0]):
    print(f"\n🔹 Result {i+1}:")
    print(doc[:300])
    print("...")

⏳ Setting up ChromaDB vector store...

✅ ChromaDB collection created!

⏳ Storing 102 chunks in ChromaDB...
✅ All chunks stored successfully!
📦 Total items in ChromaDB: 102

🔍 Running a test search: 'what is retrieval augmented generation?'

--- TOP 3 MOST RELEVANT CHUNKS FOUND ---

🔹 Result 1:
and non-parametric memory to the “workhorse of NLP,” i.e. sequence-to-sequence (seq2seq) models.
We endow pre-trained, parametric-memory generation models with a non-parametric memory through
a general-purpose ﬁne-tuning approach which we refer to as retrieval-augmented generation (RAG).
We build RA
...

🔹 Result 2:
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric
memory have so far been only investigated for extract
...

🔹 Result 3:
and generative tasks. Our work aims to expand the space of possible t

In [15]:
# ============================================================
# CELL 9 — Save Progress Checkpoint
# ============================================================
import pickle
import os

# Create a checkpoints folder
os.makedirs("checkpoints", exist_ok=True)

# Save the chunks
with open("checkpoints/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

# Save the embeddings
with open("checkpoints/embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

# Save chunk texts separately
with open("checkpoints/chunk_texts.pkl", "wb") as f:
    pickle.dump(chunk_texts, f)

print("✅ Checkpoint saved!")
print(f"   → chunks.pkl        ({len(chunks)} chunks)")
print(f"   → embeddings.pkl    ({embeddings.shape})")
print(f"   → chunk_texts.pkl   ({len(chunk_texts)} texts)")
print()
print("💡 If Colab disconnects, run only Cells 1-3 to reinstall")
print("   libraries, then load from checkpoint instead of")
print("   reprocessing the PDF from scratch.")

✅ Checkpoint saved!
   → chunks.pkl        (102 chunks)
   → embeddings.pkl    ((102, 384))
   → chunk_texts.pkl   (102 texts)

💡 If Colab disconnects, run only Cells 1-3 to reinstall
   libraries, then load from checkpoint instead of
   reprocessing the PDF from scratch.


In [20]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted!")

Mounted at /content/drive
✅ Drive mounted!


In [21]:
import json
import glob

# Find the notebook in Google Drive
notebooks = glob.glob("/content/drive/MyDrive/**/*.ipynb", recursive=True)
print("Found notebooks:")
for nb in notebooks:
    print(f"  {nb}")

Found notebooks:
  /content/drive/MyDrive/Colab Notebooks/pdf-chatbot-capstone.ipynb
  /content/drive/MyDrive/Colab Notebooks/Copy of TimesPro_Numpy_and_pandas_fundamentals (1).ipynb
  /content/drive/MyDrive/Colab Notebooks/Copy of TimesPro_Numpy_and_pandas_fundamentals.ipynb
  /content/drive/MyDrive/Colab Notebooks/Credit_Card_Limit_Prediction_Enhanced.ipynb
  /content/drive/MyDrive/Colab Notebooks/credit_card_limit_prediction_date (1).ipynb
  /content/drive/MyDrive/Colab Notebooks/credit_card_limit_prediction_date.ipynb
  /content/drive/MyDrive/Colab Notebooks/Credit_Card_Limit_Prediction_Best_in_Class.ipynb
  /content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
  /content/drive/MyDrive/Colab Notebooks/Kishore-Credit_Card_Limit_Prediction.ipynb
  /content/drive/MyDrive/Colab Notebooks/Kishore_Car_Damage_CNN.ipynb
  /content/drive/MyDrive/Colab Notebooks/Kishore_Car_Damage_CNN_Final.ipynb
  /content/drive/MyDrive/Colab Notebooks/topper--credit_limit_prediction_linear_regression.ipyn

In [22]:
import json

# Path to your notebook
notebook_path = "/content/drive/MyDrive/Colab Notebooks/pdf-chatbot-capstone.ipynb"

# Read the notebook
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Fix 1 — Remove widget metadata from notebook level
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]
    print("✅ Removed widget metadata from notebook level")
else:
    print("ℹ️ No widget metadata at notebook level")

# Fix 2 — Remove widget outputs from all cells
cleaned = 0
for cell in nb.get("cells", []):
    new_outputs = []
    for output in cell.get("outputs", []):
        if output.get("output_type") == "display_data":
            if "application/vnd.jupyter.widget-view+json" in output.get("data", {}):
                cleaned += 1
                continue  # skip widget outputs
        new_outputs.append(output)
    cell["outputs"] = new_outputs

print(f"✅ Removed {cleaned} widget outputs from cells")

# Save fixed notebook back to Drive
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

print("✅ Fixed notebook saved to Google Drive!")
print()
print("Now do: File → Save a copy in GitHub")
print("Same repo, same path → GitHub will render correctly!")

✅ Removed widget metadata from notebook level
✅ Removed 13 widget outputs from cells
✅ Fixed notebook saved to Google Drive!

Now do: File → Save a copy in GitHub
Same repo, same path → GitHub will render correctly!


In [23]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/pdf-chatbot-capstone.ipynb"

# Read the notebook
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# AGGRESSIVE FIX — strip ALL metadata from notebook level
# Keep only essential keys
clean_metadata = {}
if "kernelspec" in nb.get("metadata", {}):
    clean_metadata["kernelspec"] = nb["metadata"]["kernelspec"]
if "language_info" in nb.get("metadata", {}):
    clean_metadata["language_info"] = nb["metadata"]["language_info"]

nb["metadata"] = clean_metadata
print("✅ Stripped all non-essential notebook metadata")

# AGGRESSIVE FIX — clear ALL outputs from ALL cells completely
for cell in nb.get("cells", []):
    cell["outputs"] = []
    cell["execution_count"] = None

print("✅ Cleared all cell outputs and execution counts")

# Save
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

print("✅ Clean notebook saved!")
print()
print("Now: File → Save a copy in GitHub → same repo and path")

✅ Stripped all non-essential notebook metadata
✅ Cleared all cell outputs and execution counts
✅ Clean notebook saved!

Now: File → Save a copy in GitHub → same repo and path
